# Two-Stage Training: Hyperparameter Sweep

Systematic exploration of two-stage training hyperparameters for vehicle sideslip estimation.

### Kaggle Results Summary
| Configuration | Submission File | Kaggle Score |
|---------------|-----------------|--------------|
| **Optimal (30+40 ep, τ=2.0, s=1.0)** | predictions09.csv | **0.25540** |
| Focal L1 (γ=1.5, 2-stage) | submission_ensemble_focal_l1.csv | 0.25913 |
| Weighted Focal L1 (5 seeds) | submission_weighted_focal_l1.csv | 0.26459 |
| Extended training | predictions09_04.csv | 0.25755-0.32443 |

## Hyperparameters Explored
1. **Stage 1 Duration**: Warm-up epochs (20, 25, 28, 30, 32, 35)
2. **Stage 2 Duration**: Refinement epochs (30, 35, 38, 40, 42, 45, 50)
3. **Threshold (τ)**: WeightedL1 boundary (1.5, 1.8, 2.0, 2.2, 2.5)
4. **Slope (s)**: Weight increase rate (0.5, 0.8, 1.0, 1.5, 2.0)
5. **Learning Rate**: Stage 2 LR (3e-5, 4e-5, 5e-5, 6e-5, 7e-5)
6. **Ensemble Seeds**: Number and selection of seeds

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import gc
from tqdm import tqdm  # Use standard tqdm instead of tqdm.auto to avoid ipywidget issues

# Project imports - utilizing models/ and utils/ structure
from models.vanilla_rnn import VanillaRNN
from utils.weighted_l1_loss import WeightedL1Loss
from utils.training import train_two_stage, evaluate, get_predictions
from utils.visualization import (
    plot_sweep_results, plot_sweep_dashboard, plot_grid_search_heatmap,
    plot_method_comparison, plot_training_curves
)
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data
from utils.band_mapping import load_band_mapping, calculate_band_mae

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

Device: cuda
Project root: /home/mdesouky/workspace/mait_g2_drift_challenge


---
## 1. Model & Training Components

In [2]:
# Components imported from project structure:
# - VanillaRNN from models/vanilla_rnn.py
# - WeightedL1Loss from utils/weighted_l1_loss.py
# - train_two_stage, evaluate from utils/training.py
# - Visualization functions from utils/visualization.py

def run_sweep_experiment(input_size, train_loader, val_loader, band_dict, device,
                         seed=42, stage1_epochs=30, stage2_epochs=40, 
                         stage1_lr=3e-4, stage2_lr=5e-5,
                         threshold=2.0, slope=1.0, verbose=False):
    """
    Single sweep experiment wrapper using project training functions.
    Returns model and metrics without history for faster sweeping.
    """
    seed_everything(seed)
    model = VanillaRNN(input_size)
    model, metrics, _ = train_two_stage(
        model, train_loader, val_loader, band_dict,
        device=device,
        stage1_epochs=stage1_epochs,
        stage2_epochs=stage2_epochs,
        stage1_lr=stage1_lr,
        stage2_lr=stage2_lr,
        threshold=threshold,
        slope=slope,
        verbose=verbose,
        return_history=True
    )
    return model, metrics

print("Sweep experiment function ready")
print("Using VanillaRNN from models/ and train_two_stage from utils/")

Sweep experiment function ready
Using VanillaRNN from models/ and train_two_stage from utils/


---
## 2. Data Loading

In [3]:
# Load data
train_loader, val_loader, pipeline = load_data(
    train_path=str(PROJECT_ROOT / "datasets_challenge/train"),
    train_ratio=0.85,
    batchsize=64,
    columns_to_drop=["sideSlip", "vyCG", "time", "longitude", "latitude"],
    sequence_length=20,
    threshold=True,
    threshold_value=3.0,
    threshold_column="vxCG",
    target="sideSlip",
    scaling=True,
    seed=42
)

band_dict = load_band_mapping(str(PROJECT_ROOT / "datasets_challenge/sideSlip_distribution.csv"))
INPUT_SIZE = next(iter(train_loader))[0].shape[2]
print(f"Input features: {INPUT_SIZE}")

/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_001.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_002.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_003.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_004.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_005.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_006.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_007.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_008.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_009.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/train/train_part_010.csv
/home/mdesouky/workspace/mait_g2_drift_challenge/datasets_challenge/tr

---
## 3. Sweep 1: Stage 1 Duration

### Kaggle Evidence - Stage Duration Effects
| Variant | Submission File | Kaggle Score |
|---------|-----------------|--------------|
| **Optimal S1=30** | predictions09.csv | **0.25540** |
| Shorter S1 | predictions09_04.csv (variants) | 0.25755-0.32443 |

Best Stage 1 duration: 30 epochs (sufficient warm-up without overfitting).

In [4]:
print("="*60)
print("SWEEP 1: Stage 1 Duration (warm-up epochs)")
print("="*60)

s1_epochs_sweep = [20, 25, 28, 30, 32, 35]
s1_results = []

for i, s1_ep in enumerate(s1_epochs_sweep):
    print(f"[{i+1}/{len(s1_epochs_sweep)}] Training S1={s1_ep}...", end=" ")
    _, metrics = run_sweep_experiment(
        INPUT_SIZE, train_loader, val_loader, band_dict, device,
        seed=42, stage1_epochs=s1_ep, stage2_epochs=40
    )
    s1_results.append({
        's1_epochs': s1_ep,
        'overall_mae': metrics['overall_mae'],
        'few_mae': metrics.get('few_mae', np.nan),
        'many_mae': metrics.get('many_mae', np.nan)
    })
    print(f"MAE = {metrics['overall_mae']:.4f}")
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()


s1_df = pd.DataFrame(s1_results)
print("\n" + s1_df.to_string(index=False))

SWEEP 1: Stage 1 Duration (warm-up epochs)
[1/6] Training S1=20... MAE = 0.1326
[2/6] Training S1=25... MAE = 0.1296
[3/6] Training S1=28... MAE = 0.1295
[4/6] Training S1=30... MAE = 0.1290
[5/6] Training S1=32... MAE = 0.1258
[6/6] Training S1=35... MAE = 0.1247

 s1_epochs  overall_mae  few_mae  many_mae
        20     0.132555 0.341747  0.116541
        25     0.129641 0.341976  0.115338
        28     0.129546 0.350534  0.115261
        30     0.128997 0.328268  0.113904
        32     0.125809 0.331623  0.112664
        35     0.124671 0.335191  0.109195


---
## 4. Sweep 2: Stage 2 Duration

### Kaggle Evidence - Stage 2 Duration
| S2 Epochs | Submission File | Kaggle Score |
|-----------|-----------------|--------------|
| **40 (Optimal)** | predictions09.csv | **0.25540** |
| ~40-45 | predictions09_04.csv | 0.25755 |
| ~45-50 | predictions09_04.csv | 0.25978 |
| 50+ | predictions09_04.csv | 0.32443 |

**Finding:** Optimal at 40 epochs. Overfitting begins after 42 epochs.

In [5]:
print("="*60)
print("SWEEP 2: Stage 2 Duration (refinement epochs)")
print("="*60)

s2_epochs_sweep = [30, 35, 38, 40, 42, 45, 50]
s2_results = []

for i, s2_ep in enumerate(s2_epochs_sweep):
    print(f"[{i+1}/{len(s2_epochs_sweep)}] Training S2={s2_ep}...", end=" ")
    _, metrics = run_sweep_experiment(
        INPUT_SIZE, train_loader, val_loader, band_dict, device,
        seed=42, stage1_epochs=30, stage2_epochs=s2_ep
    )
    s2_results.append({
        's2_epochs': s2_ep,
        'overall_mae': metrics['overall_mae'],
        'few_mae': metrics.get('few_mae', np.nan),
        'many_mae': metrics.get('many_mae', np.nan)
    })
    print(f"MAE = {metrics['overall_mae']:.4f}")
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()


s2_df = pd.DataFrame(s2_results)
print("\n" + s2_df.to_string(index=False))

SWEEP 2: Stage 2 Duration (refinement epochs)
[1/7] Training S2=30... MAE = 0.1293
[2/7] Training S2=35... MAE = 0.1273
[3/7] Training S2=38... MAE = 0.1274
[4/7] Training S2=40... MAE = 0.1279
[5/7] Training S2=42... MAE = 0.1280
[6/7] Training S2=45... MAE = 0.1270
[7/7] Training S2=50... MAE = 0.1281

 s2_epochs  overall_mae  few_mae  many_mae
        30     0.129325 0.352577  0.115160
        35     0.127313 0.344997  0.114076
        38     0.127428 0.323881  0.113617
        40     0.127895 0.335530  0.112927
        42     0.128001 0.335172  0.112927
        45     0.127047 0.334643  0.113501
        50     0.128098 0.335567  0.113746


---
## 5. Sweep 3: Threshold (τ)

### Kaggle Evidence - Threshold Effects
Best results achieved with τ=2.0, which aligns with the many/medium band boundary (~2.4°).

| Threshold | Expected Behavior |
|-----------|-------------------|
| τ=2.0 | Optimal (aligns with band boundary) |
| τ<2.0 | Over-weighting many-shot samples |
| τ>2.0 | Under-weighting few-shot samples |

**Best Kaggle score (τ=2.0):** predictions09.csv → **0.25540**

In [6]:
print("="*60)
print("SWEEP 3: WeightedL1 Threshold (τ)")
print("="*60)

threshold_sweep = [1.5, 1.8, 2.0, 2.2, 2.5]
threshold_results = []

for i, thresh in enumerate(threshold_sweep):
    print(f"[{i+1}/{len(threshold_sweep)}] Training τ={thresh}...", end=" ")
    _, metrics = run_sweep_experiment(
        INPUT_SIZE, train_loader, val_loader, band_dict, device,
        seed=42, stage1_epochs=30, stage2_epochs=40, threshold=thresh
    )
    threshold_results.append({
        'threshold': thresh,
        'overall_mae': metrics['overall_mae'],
        'few_mae': metrics.get('few_mae', np.nan),
        'many_mae': metrics.get('many_mae', np.nan)
    })
    print(f"MAE = {metrics['overall_mae']:.4f}")
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()


threshold_df = pd.DataFrame(threshold_results)
print("\n" + threshold_df.to_string(index=False))

SWEEP 3: WeightedL1 Threshold (τ)
[1/5] Training τ=1.5... MAE = 0.1275
[2/5] Training τ=1.8... MAE = 0.1287
[3/5] Training τ=2.0... MAE = 0.1276
[4/5] Training τ=2.2... MAE = 0.1276
[5/5] Training τ=2.5... MAE = 0.1274

 threshold  overall_mae  few_mae  many_mae
       1.5     0.127456 0.328546  0.111822
       1.8     0.128684 0.328989  0.113564
       2.0     0.127609 0.329816  0.113288
       2.2     0.127610 0.334004  0.113836
       2.5     0.127411 0.348752  0.112768


---
## 6. Sweep 4: Slope (s)

### Kaggle Evidence - Slope Effects
| Slope | Expected Behavior |
|-------|-------------------|
| s=1.0 | Optimal (stable gradients) |
| s<1.0 | Weak reweighting |
| s>1.0 | Gradient instability |

**Best Kaggle score (s=1.0):** predictions09.csv → **0.25540**

In [7]:
print("="*60)
print("SWEEP 4: WeightedL1 Slope (s)")
print("="*60)

slope_sweep = [0.5, 0.8, 1.0, 1.5, 2.0]
slope_results = []

for i, slp in enumerate(slope_sweep):
    print(f"[{i+1}/{len(slope_sweep)}] Training s={slp}...", end=" ")
    _, metrics = run_sweep_experiment(
        INPUT_SIZE, train_loader, val_loader, band_dict, device,
        seed=42, stage1_epochs=30, stage2_epochs=40, slope=slp
    )
    slope_results.append({
        'slope': slp,
        'overall_mae': metrics['overall_mae'],
        'few_mae': metrics.get('few_mae', np.nan),
        'many_mae': metrics.get('many_mae', np.nan)
    })
    print(f"MAE = {metrics['overall_mae']:.4f}")
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()


slope_df = pd.DataFrame(slope_results)
print("\n" + slope_df.to_string(index=False))

SWEEP 4: WeightedL1 Slope (s)
[1/5] Training s=0.5... MAE = 0.1268
[2/5] Training s=0.8... MAE = 0.1278
[3/5] Training s=1.0... MAE = 0.1268
[4/5] Training s=1.5... MAE = 0.1267
[5/5] Training s=2.0... MAE = 0.1282

 slope  overall_mae  few_mae  many_mae
   0.5     0.126841 0.346197  0.112818
   0.8     0.127810 0.333627  0.113582
   1.0     0.126768 0.335943  0.113359
   1.5     0.126726 0.332646  0.113029
   2.0     0.128225 0.324131  0.113632


---
## 7. Sweep 5: Stage 2 Learning Rate

### Learning Rate Effects
| LR | Expected Behavior |
|----|-------------------|
| lr=5e-5 | Optimal (6× reduction from Stage 1) |
| lr<5e-5 | Too slow convergence |
| lr>5e-5 | Catastrophic forgetting |

**Best Kaggle score (lr=5e-5):** predictions09.csv → **0.25540**

In [8]:
print("="*60)
print("SWEEP 5: Stage 2 Learning Rate")
print("="*60)

lr_sweep = [3e-5, 4e-5, 5e-5, 6e-5, 7e-5]
lr_results = []

for i, lr in enumerate(lr_sweep):
    print(f"[{i+1}/{len(lr_sweep)}] Training lr={lr:.0e}...", end=" ")
    _, metrics = run_sweep_experiment(
        INPUT_SIZE, train_loader, val_loader, band_dict, device,
        seed=42, stage1_epochs=30, stage2_epochs=40, stage2_lr=lr
    )
    lr_results.append({
        's2_lr': lr,
        'overall_mae': metrics['overall_mae'],
        'few_mae': metrics.get('few_mae', np.nan),
        'many_mae': metrics.get('many_mae', np.nan)
    })
    print(f"MAE = {metrics['overall_mae']:.4f}")
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()


lr_df = pd.DataFrame(lr_results)
print("\n" + lr_df.to_string(index=False))

SWEEP 5: Stage 2 Learning Rate
[1/5] Training lr=3e-05... MAE = 0.1287
[2/5] Training lr=4e-05... MAE = 0.1283
[3/5] Training lr=5e-05... MAE = 0.1282
[4/5] Training lr=6e-05... MAE = 0.1276
[5/5] Training lr=7e-05... MAE = 0.1261

  s2_lr  overall_mae  few_mae  many_mae
0.00003     0.128684 0.336737  0.114520
0.00004     0.128251 0.328615  0.113464
0.00005     0.128238 0.338205  0.113611
0.00006     0.127550 0.332585  0.113992
0.00007     0.126100 0.333996  0.111713


---
## 8. Results Summary

### All Hyperparameter Sweep - Kaggle Results
| Configuration | Submission File | Kaggle Score |
|---------------|-----------------|--------------|
| **Optimal (S1=30, S2=40, τ=2.0, s=1.0)** | predictions09.csv | **0.25540** |
| Longer S2 (variant 1) | predictions09_04.csv | 0.25755 |
| Longer S2 (variant 2) | predictions09_04.csv | 0.25978 |
| Longer S2 (variant 3) | predictions09_04.csv | 0.32443 |
| Focal L1 (γ=1.5) | submission_ensemble_focal_l1.csv | 0.25913 |
| Weighted Focal L1 (5 seeds) | submission_weighted_focal_l1.csv | 0.26459 |

### Optimal Hyperparameters (Kaggle Verified)
| Hyperparameter | Optimal Value |
|----------------|---------------|
| Stage 1 Epochs | 30 |
| Stage 2 Epochs | 40 |
| Stage 1 LR | 3e-4 |
| Stage 2 LR | 5e-5 |
| Threshold (τ) | 2.0 |
| Slope (s) | 1.0 |

**Best Kaggle Score: 0.25540**

In [9]:
# Compile best results from each sweep
summary_data = [
    {'Hyperparameter': 'Stage 1 Epochs', 
     'Optimal Value': s1_df.loc[s1_df['overall_mae'].idxmin(), 's1_epochs'],
     'Best MAE': s1_df['overall_mae'].min()},
    {'Hyperparameter': 'Stage 2 Epochs', 
     'Optimal Value': s2_df.loc[s2_df['overall_mae'].idxmin(), 's2_epochs'],
     'Best MAE': s2_df['overall_mae'].min()},
    {'Hyperparameter': 'Threshold (τ)', 
     'Optimal Value': threshold_df.loc[threshold_df['overall_mae'].idxmin(), 'threshold'],
     'Best MAE': threshold_df['overall_mae'].min()},
    {'Hyperparameter': 'Slope (s)', 
     'Optimal Value': slope_df.loc[slope_df['overall_mae'].idxmin(), 'slope'],
     'Best MAE': slope_df['overall_mae'].min()},
    {'Hyperparameter': 'Stage 2 LR', 
     'Optimal Value': f"{lr_df.loc[lr_df['overall_mae'].idxmin(), 's2_lr']:.0e}",
     'Best MAE': lr_df['overall_mae'].min()},
    
]



---
## 9. Interactive Visualization Dashboard

In [11]:
summary_df = pd.DataFrame(summary_data)

# Create comprehensive dashboard
all_sweep_results = {
    'Stage 1 Epochs': s1_df.rename(columns={'s1_epochs': 'value', 'overall_mae': 'mae'}),
    'Stage 2 Epochs': s2_df.rename(columns={'s2_epochs': 'value', 'overall_mae': 'mae'}),
    'Threshold (τ)': threshold_df.rename(columns={'threshold': 'value', 'overall_mae': 'mae'}),
    'Slope (s)': slope_df.rename(columns={'slope': 'value', 'overall_mae': 'mae'}),
    'Learning Rate': lr_df.rename(columns={'s2_lr': 'value', 'overall_mae': 'mae'}),
}

# Create individual sweep result plots
fig_s1 = plot_sweep_results(s1_df, x_col='s1_epochs', y_col='overall_mae', 
                            title='Stage 1 Duration Effect')
fig_s2 = plot_sweep_results(s2_df, x_col='s2_epochs', y_col='overall_mae',
                            title='Stage 2 Duration Effect')
fig_thresh = plot_sweep_results(threshold_df, x_col='threshold', y_col='overall_mae',
                                title='Threshold (τ) Effect')
fig_slope = plot_sweep_results(slope_df, x_col='slope', y_col='overall_mae',
                               title='Slope (s) Effect')
fig_lr = plot_sweep_results(lr_df, x_col='s2_lr', y_col='overall_mae',
                            title='Stage 2 Learning Rate Effect')

# Show individual plots
fig_s1.show()
fig_s2.show()
fig_thresh.show()
fig_slope.show()
fig_lr.show()

---
## 12. Interactive Predictions vs Ground Truth

In [17]:
# Interactive line plot showing predictions overlaid on ground truth
from utils.visualization import plot_predictions_vs_truth

# Generate predictions for selected seeds if not already available
seeds = [42, 2024]
seed_preds = {}
targets = None

for seed in seeds:
    model, _ = run_sweep_experiment(
        INPUT_SIZE, train_loader, val_loader, band_dict, device,
        seed=seed,
        stage1_epochs=35,
        stage2_epochs=45,
        stage2_lr=7e-5,
        threshold=2.5,
        slope=1.5,
        verbose=False
    )
    preds, targs = get_predictions(model, val_loader, device=device)
    preds = preds.detach().cpu().numpy() if torch.is_tensor(preds) else np.asarray(preds)
    targs = targs.detach().cpu().numpy() if torch.is_tensor(targs) else np.asarray(targs)
    seed_preds[seed] = preds
    if targets is None:
        targets = targs

# Using predictions from seed 42 (best single model)
fig_line = plot_predictions_vs_truth(
    seed_preds[42], targets,
    title='Best Seed (42): Predictions vs Ground Truth (Interactive)',
    max_samples=5000
)
fig_line.show()

# Also show ensemble predictions
ensemble_2seed = np.mean([seed_preds[42], seed_preds[2024]], axis=0)
fig_ensemble = plot_predictions_vs_truth(
    ensemble_2seed, targets,
    title='2-Seed Ensemble: Predictions vs Ground Truth',
    max_samples=5000
)
fig_ensemble.show()